# 05 - Achievements table

- First look at `STEAM_DB.STEAM_RAW.ACHIEVEMENTS`.
- It comes from `ISteamUserStats/GetGlobalAchievementPercentagesForApp`. No API key needed.
- One row is one achievement of one game in one run: `achievement_name` and `percent`. The percent is the share of players who unlocked it. Steam sends it as **text** (like `"40.9"`), and I keep it as text.
- The name is a code like `achievement_economy_1`, not a readable title. Readable titles need a Steam key, and I don't use one.
- A game with no data gets **no rows**, not a row with 0 %. For unreleased games and wrong ids, Steam answers HTTP 403 with an empty `{}`.
- `collected_at` is one time for the whole run.

- The data is one load from 2026-11-02 (load id `1793621677.9089065`). The run asked 81 games. The log said: `achievements: 49 games with rows (3966 rows), 32 without data, 0 skipped, 0 failed`.
- The notebook only runs SELECT and DESCRIBE, so it changes nothing in Snowflake.

- Limits:
  - There is only one snapshot, so no change over time yet.
  - The percentages move slowly, so later runs will repeat mostly the same numbers.
  - I don't know what the percent is measured against (all owners or all players).
  - I don't know if achievements of DLC are in the list.
  - Valve does not document this endpoint.

## 0. Connect to Snowflake

Same connection as notebooks 01 to 04. It reads the settings from `.env`, logs in as `STEAM_LOADER` and prints no secrets. Times are in UTC (Stockholm is UTC+1 now). Pick the project's `.venv` as the kernel.

In [1]:
import os
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [2]:
def find_repo_root() -> Path:
    """Walk up from the notebook's folder until we find pyproject.toml."""
    here = Path.cwd().resolve()
    return next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())


def load_env_file(path: Path) -> None:
    """Copy KEY=value lines from a .env file into os.environ (never prints values)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env_file(find_repo_root() / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_LOADER_USER"],
    private_key_file=os.environ["SNOWFLAKE_LOADER_PRIVATE_KEY_PATH"],
    role=os.environ["SNOWFLAKE_LOADER_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="STEAM_RAW",
    session_parameters={"TIMEZONE": "UTC"},
)


def query(sql: str) -> pd.DataFrame:
    """Run one read-only statement and return the result as a DataFrame."""
    with conn.cursor() as cursor:
        cursor.execute(sql)
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)

In [3]:
# Sanity check: we should be the loader role, in STEAM_DB.STEAM_RAW.
query("""
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema
""")

,ROLE,WAREHOUSE,DATABASE,SCHEMA
0,STEAM_LOADER,STEAM_WH,STEAM_DB,STEAM_RAW


In [4]:
# The newest load. The queries below use it, so they keep working when more runs arrive.
LOAD_ID = query("SELECT MAX(_dlt_load_id) AS load_id FROM achievements")["LOAD_ID"][0]
LOAD_ID

'1793621677.9089065'

## 1. Rows per load

- Every run adds one row per game and achievement. All rows of one run share one `collected_at`.

**💡Insights:**
- One load, 3,966 rows, from 49 games. The log said the same.
- All rows have the same `collected_at` (12:14:42 UTC).
- There are only 3,925 different names, so 41 names are used by more than one game. The achievement name alone is not a key. We need the pair (`steam_appid`, `achievement_name`).
- The second query below lists those 41 names. Each one is used by exactly 2 games (41 names x 1 extra game = the 41 missing names). Many look like they come from game pairs, for example `Class Barbarian`, `Class Druid` and `Class Monk`, which sound like the two *Knights of Pen and Paper* games. I did not check which games share them.

In [5]:
query("""
SELECT _dlt_load_id, COUNT(*) AS rows_loaded, COUNT(DISTINCT steam_appid) AS distinct_games,
       COUNT(DISTINCT achievement_name) AS distinct_names,
       MIN(collected_at) AS first_row, MAX(collected_at) AS last_row
FROM achievements
GROUP BY _dlt_load_id
ORDER BY _dlt_load_id
""")

,_DLT_LOAD_ID,ROWS_LOADED,DISTINCT_GAMES,DISTINCT_NAMES,FIRST_ROW,LAST_ROW
0,1793621677.9089065,3966,49,3925,2026-11-02 12:14:42.518944+00:00,2026-11-02 12:14:42.518944+00:00


In [6]:
# Names that more than one game uses.
query(f"""
SELECT achievement_name, COUNT(DISTINCT steam_appid) AS games
FROM achievements
WHERE _dlt_load_id = '{LOAD_ID}'
GROUP BY achievement_name
HAVING COUNT(DISTINCT steam_appid) > 1
ORDER BY games DESC, achievement_name
""")

,ACHIEVEMENT_NAME,GAMES
0,ACH_WIN_FIRST,2
1,Bearded Lady,2
2,Bring It On,2
3,Champion,2
4,Class Barbarian,2
5,Class Druid,2
6,Class Knight,2
7,Class Monk,2
8,Class Ninja,2
9,Class Psion,2


## 2. Duplicates

One achievement should show up once per game and run. This lists any that show up twice.

**💡Insights:**
- The result is empty, so there are no duplicates.

In [7]:
query("""
SELECT steam_appid, achievement_name, _dlt_load_id, COUNT(*) AS copies
FROM achievements
GROUP BY steam_appid, achievement_name, _dlt_load_id
HAVING COUNT(*) > 1
""")

,STEAM_APPID,ACHIEVEMENT_NAME,_DLT_LOAD_ID,COPIES


## 3. Column types

**💡Insights:**
- `PERCENT` is text (`VARCHAR`), as intended. The `columns` hint on the resource fixes the type, so a number that shows up one day cannot split the column in two.
- `ACHIEVEMENT_NAME` is also text. `STEAM_APPID` is `NUMBER(19,0)` and `COLLECTED_AT` is `TIMESTAMP_LTZ(9)`.
- dlt added `_DLT_LOAD_ID` and `_DLT_ID`. Every other column can be NULL.

In [8]:
query("DESCRIBE TABLE achievements")[["name", "type", "null?"]]

,name,type,null?
0,COLLECTED_AT,TIMESTAMP_LTZ(9),Y
1,PERCENT,VARCHAR(16777216),Y
2,STEAM_APPID,"NUMBER(19,0)",Y
3,ACHIEVEMENT_NAME,VARCHAR(16777216),Y
4,_DLT_LOAD_ID,VARCHAR(16777216),N
5,_DLT_ID,VARCHAR(16777216),N


## 4. Is the text a number?

- `TRY_TO_DOUBLE` turns the text into a number, or gives NULL if it cannot. So `not_a_number` counts the bad values.

**💡Insights:**
- All 3,966 values are numbers (`not_a_number` is 0).
- The lowest is 0.0 and the highest is 100.0, so everything stays between 0 and 100.
- A 0.0 is what Steam shows at one decimal. I don't know the real count behind it.

In [9]:
query(f"""
SELECT COUNT(*) AS rows_checked,
       COUNT_IF(TRY_TO_DOUBLE(percent) IS NULL) AS not_a_number,
       MIN(TRY_TO_DOUBLE(percent)) AS min_percent,
       MAX(TRY_TO_DOUBLE(percent)) AS max_percent
FROM achievements
WHERE _dlt_load_id = '{LOAD_ID}'
""")

,ROWS_CHECKED,NOT_A_NUMBER,MIN_PERCENT,MAX_PERCENT
0,3966,0,0.0,100.0


## 5. Games without achievement rows

- The run asked 81 games. 49 gave rows. The other 32 gave no data, either HTTP 403 with `{}` or an empty list. The log does not say which.
- This query lists those 32 games.

**💡Insights:**
- 4 of the 32 are unreleased (`coming_soon` is true): Afterworld, Guardians of the Wild Sky, LEGO Skylines and Prison Architect 2.
- The other 28 are released, and all of them came out between 2008 and 2013. Some examples: *Victoria II*, *Hearts of Iron III*, *Knights of Honor*, *Shadowrun Returns* and *March of the Eagles*.
- I can't say yet that age is the reason, because I did not look at the release dates of the 49 games that have rows.
- The next section compares the 32 with the store page.

In [10]:
query(f"""
SELECT g.name, g.coming_soon, g.release_date_text
FROM games g
WHERE g._dlt_load_id = '{LOAD_ID}'
  AND g.steam_appid NOT IN (
      SELECT steam_appid FROM achievements WHERE _dlt_load_id = '{LOAD_ID}'
  )
ORDER BY g.coming_soon DESC, g.name
""")

,NAME,COMING_SOON,RELEASE_DATE_TEXT
0,Afterworld,True,To be announced
1,Guardians of the Wild Sky,True,Coming soon
2,LEGO® Skylines,True,Coming soon
3,Prison Architect 2,True,To be announced
4,Age of Wonders,False,"12 Oct, 2010"
5,Age of Wonders II: The Wizard's Throne,False,"12 Oct, 2010"
6,Age of Wonders Shadow Magic,False,"12 Oct, 2010"
7,Arsenal of Democracy: A Hearts of Iron Game,False,"23 Feb, 2010"
8,Crusader Kings Complete,False,"14 Feb, 2012"
9,Crusaders: Thy Kingdom Come,False,"8 Oct, 2009"


## 6. Cross-check with the store page

- The `games` table has the store categories. A game with Steam achievements should have the category `Steam Achievements`.
- If our table agrees with the store, then the games without rows really have no achievements.

**💡Insights:**

| We have rows | Store lists achievements | Games |
|:---:|:---:|---:|
| no | no | 30 |
| no | yes | 2 |
| yes | no | 5 |
| yes | yes | 44 |

- 74 of 81 games agree with the store (30 + 44).
- The 2 games with the category but no rows are *Afterworld* and *Prison Architect 2*. Both are unreleased, so Steam answers 403 for them. No data was lost. Nothing can be counted before release.
- The 5 games with rows but no category are all released: *A Game of Dwarves* (2012), *Crusader Kings II* (2012), *FOUNDRY* (2024), *Star Trek: Infinite* (2023) and *Sword of the Stars II: Enhanced Edition* (2012). I don't know why the store category is missing for them.
- So the store category is not a perfect check. For a `has_achievements` flag we should use our own rows, not the category.
- The 28 released games without rows also have no category, so nothing points to lost data there.
- The second query below lists the 7 games that disagree.

In [11]:
query(f"""
SELECT a.steam_appid IS NOT NULL AS has_rows,
       ARRAY_CONTAINS('Steam Achievements'::VARIANT, g.categories) AS store_says_achievements,
       COUNT(*) AS games
FROM games g
LEFT JOIN (
    SELECT DISTINCT steam_appid FROM achievements WHERE _dlt_load_id = '{LOAD_ID}'
) a ON a.steam_appid = g.steam_appid
WHERE g._dlt_load_id = '{LOAD_ID}'
GROUP BY has_rows, store_says_achievements
ORDER BY has_rows, store_says_achievements
""")

,HAS_ROWS,STORE_SAYS_ACHIEVEMENTS,GAMES
0,False,False,30
1,False,True,2
2,True,False,5
3,True,True,44


In [12]:
# The games where our table and the store do not agree.
query(f"""
WITH checked AS (
    SELECT g.name, g.release_date_text,
           a.steam_appid IS NOT NULL AS has_rows,
           ARRAY_CONTAINS('Steam Achievements'::VARIANT, g.categories) AS store_says_achievements
    FROM games g
    LEFT JOIN (
        SELECT DISTINCT steam_appid FROM achievements WHERE _dlt_load_id = '{LOAD_ID}'
    ) a ON a.steam_appid = g.steam_appid
    WHERE g._dlt_load_id = '{LOAD_ID}'
)
SELECT *
FROM checked
WHERE has_rows != store_says_achievements
ORDER BY has_rows, name
""")

,NAME,RELEASE_DATE_TEXT,HAS_ROWS,STORE_SAYS_ACHIEVEMENTS
0,Afterworld,To be announced,False,True
1,Prison Architect 2,To be announced,False,True
2,A Game of Dwarves,"23 Oct, 2012",True,False
3,Crusader Kings II,"14 Feb, 2012",True,False
4,FOUNDRY,"2 May, 2024",True,False
5,Star Trek: Infinite,"12 Oct, 2023",True,False
6,Sword of the Stars II: Enhanced Edition,"30 Nov, 2012",True,False


## 7. Achievements per game

**💡Insights:**

| Game | Achievements |
|---|---:|
| Europa Universalis IV | 373 |
| Hearts of Iron IV | 293 |
| Stellaris | 219 |
| Crusader Kings III | 198 |
| Crusader Kings II | 161 |
| ... | |
| Prison Architect | 18 |
| Sword of the Stars II: Enhanced Edition | 14 |
| NECROPOLIS: BRUTAL EDITION | 13 |

- The top five are big strategy games with a lot of DLC. I did not check if DLC achievements are in these counts.
- The 49 games range from 13 to 373 achievements.
- EU4, HoI4, Stellaris and CK3 together have 1,083 of the 3,966 rows (my sum), about 27 %.
- Stellaris has 219, the same as in the first probe.

In [13]:
query(f"""
SELECT g.name, COUNT(*) AS achievements
FROM achievements a
JOIN games g ON g.steam_appid = a.steam_appid AND g._dlt_load_id = a._dlt_load_id
WHERE a._dlt_load_id = '{LOAD_ID}'
GROUP BY g.name
ORDER BY achievements DESC
""")

,NAME,ACHIEVEMENTS
0,Europa Universalis IV,373
1,Hearts of Iron IV,293
2,Stellaris,219
3,Crusader Kings III,198
4,Crusader Kings II,161
5,Across the Obelisk,148
6,Victoria 3,141
7,Academia : School Simulator,140
8,Cities: Skylines,135
9,BATTLETECH,128


## 8. How hard are the achievements?

- For each game: the average percent, the easiest achievement and the hardest.
- The average of the percents is not "the share of players who finish the game". It only says how hard the achievements are on average.

**💡Insights:**

| Lowest average | Avg % | Highest average | Avg % |
|---|---:|---|---:|
| Imperator: Rome | 1.7 | NECROPOLIS: BRUTAL EDITION | 42.4 |
| Hearts of Iron IV | 1.9 | Knights of Pen and Paper 2 | 36.7 |
| Crusader Kings II | 2.5 | Across the Obelisk | 34.7 |
| Star Trek: Infinite | 2.9 | Sword of the Stars II | 29.7 |
| Europa Universalis IV | 3.7 | Victoria 3 | 29.1 |

- Games with many achievements tend to have a low average, because most of their achievements are rare.
- Three of the five highest averages come from short lists (13 to 33 achievements). *Across the Obelisk* (148) and *Victoria 3* (141) are high even with long lists.
- *Victoria 3* has an achievement at 100 %. My guess is that you get it just by playing. Not checked.
- In *Star Trek: Infinite* even the easiest achievement is only 8.6 %. Either few players get far, or the percent is measured against something different. I don't know which.
- Five games have a hardest achievement of 0.0: Stellaris, Europa Universalis V, Cities: Skylines, Millennia and Transport Fever 3.

In [14]:
query(f"""
SELECT g.name,
       COUNT(*) AS achievements,
       ROUND(AVG(TRY_TO_DOUBLE(a.percent)), 1) AS avg_percent,
       MAX(TRY_TO_DOUBLE(a.percent)) AS easiest_percent,
       MIN(TRY_TO_DOUBLE(a.percent)) AS hardest_percent
FROM achievements a
JOIN games g ON g.steam_appid = a.steam_appid AND g._dlt_load_id = a._dlt_load_id
WHERE a._dlt_load_id = '{LOAD_ID}'
GROUP BY g.name
ORDER BY avg_percent
""")

,NAME,ACHIEVEMENTS,AVG_PERCENT,EASIEST_PERCENT,HARDEST_PERCENT
0,Imperator: Rome,72,1.7,19.2,0.3
1,Hearts of Iron IV,293,1.9,19.8,0.1
2,Crusader Kings II,161,2.5,14.4,1.2
3,Star Trek: Infinite,50,2.9,8.6,0.4
4,Europa Universalis IV,373,3.7,28.4,0.3
5,Teleglitch: Die More Edition,39,5.0,40.7,1.6
6,Stellaris,219,5.4,40.9,0.0
7,Race Arcade,28,6.0,30.1,1.7
8,Ice Lakes,60,6.3,76.7,0.7
9,Cities: Skylines,135,6.6,74.2,0.0


## 9. One game up close: Stellaris

**💡Insights:**
- The names are codes. The easiest Stellaris achievements are `achievement_economy_1` (40.9), `achievement_economy_2` (36.7), `achievement_colonize` (36.5), `achievement_rare` (31.2) and `achievement_survey` (25.4).
- Steam sends them already sorted from most to least unlocked, and the table gives the same order.
- One of the top 15 has a typo that comes from Steam: `achieviement_conquer_homeworld`. So I can't just cut the `achievement_` prefix to make a readable name. It would miss this one.
- For a dashboard the readable name is only for display. The real name stays as it is.

In [15]:
query(f"""
SELECT achievement_name, percent, TRY_TO_DOUBLE(percent) AS percent_number
FROM achievements
WHERE _dlt_load_id = '{LOAD_ID}' AND steam_appid = 281990
ORDER BY percent_number DESC
LIMIT 15
""")

,ACHIEVEMENT_NAME,PERCENT,PERCENT_NUMBER
0,achievement_economy_1,40.9,40.9
1,achievement_economy_2,36.7,36.7
2,achievement_colonize,36.5,36.5
3,achievement_rare,31.2,31.2
4,achievement_survey,25.4,25.4
5,achievement_archaeologist,25.0,25.0
6,achievement_robots,24.2,24.2
7,achievement_minerals_1,24.1,24.1
8,achievement_trade,22.8,22.8
9,achievement_winwar,22.7,22.7


## 10. Findings for dbt staging

- These are proposals for when we build the staging model. The SQL is a sketch, not tested. The model and column names are only suggestions.

---

### Finding 1: one row per game, achievement and load (staging)

- 3,966 rows, no duplicates (sections 1 and 2).
- 41 names are used by more than one game, each by exactly 2, so the name alone is not unique.
- Keep `steam_appid`, `achievement_name`, `percent` and `collected_at`. Rename `_dlt_load_id` to `load_id`.
- Tests: `unique_combination_of_columns` on (`steam_appid`, `achievement_name`, `load_id`), `not_null` on the columns, and `relationships` to the games model.

### Finding 2: the percent is text, dbt makes it a number (staging)

- All 3,966 values can be read as numbers, between 0 and 100 (section 4).
- Add `percent_unlocked` as a number and keep the original text too.
- Tests: `not_null` on `percent_unlocked` (it catches a value that is not a number) and `dbt_utils.accepted_range` from 0 to 100.

```sql
try_to_double(percent) as percent_unlocked
```

### Finding 3: no rows means no data, never 0 % (intermediate)

- 32 of 81 games have no rows (section 5).
- Do not fill anything in staging. In intermediate, start from `games`, join the achievements on the left and add a flag `has_achievements`. The dashboard shows "no data".
- Make the flag from our own rows, not from the store category. The category is missing for 5 games that do have rows (section 6).
- The 2 games with the category but no rows are unreleased, so no data was lost.

### Finding 4: the names are codes (marts)

- Keep `achievement_name` as it is.
- Steam has a typo in its own name (`achieviement_conquer_homeworld`), so a simple cut of the prefix is not safe.
- If the dashboard needs readable names, make a display name in the marts and check it by eye.

### Finding 5: repeated runs give mostly the same rows (intermediate)

- We have one load. Percentages move slowly, so daily runs will repeat almost the same numbers. A game that adds achievements shows up with new names.
- For "now", use only the latest load that has the game (the same `QUALIFY ROW_NUMBER()` idea as in the other notebooks). Keep the older loads to see changes.

### Finding 6: what to show per game (marts)

- Games have from 13 to 373 achievements, and the average percent depends a lot on that number (section 8).
- One row per game: number of achievements, average, median, easiest, hardest and the share under 5 %. Show them side by side, and do not rank by one number alone.
- Which numbers the dashboard shows is not decided.

---

**Still open**

| Question | Status |
|---|---|
| Why 5 released games have rows but no store category | Unknown |
| For the 32 games without rows: HTTP 403 or an empty list? | The log does not say. The result is the same (no rows). |
| What is the percent measured against? | Unknown |
| Are DLC achievements in the lists? | Unknown |
| Does a percent change between runs? | Needs more runs |
| Readable achievement titles | Need a Steam key, which I don't use |
| Rule for repeated runs (one per day?) | Proposal, not decided |

In [16]:
# conn.close()